# Continuous systems — exercise lab

This laboratory accompanies [the chapter](https://mvreeuwijk.github.io/chaosbook/cont1d.html). It runs
entirely in your browser (Python via Pyodide) — nothing to install, and any
changes you make are yours alone: re-opening the lab from the chapter resets it.

The full text of the exercises is in the
[chapter's exercise section](https://mvreeuwijk.github.io/chaosbook/cont1d.html#exercises); this notebook
gives you a running start on each one.

In [ ]:
# Run this cell first: it installs the book's package into the
# in-browser Python (takes a few seconds, needs to run once per visit).
%pip install -q chaosbook ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, fsolve
from ipywidgets import interact, FloatSlider
import chaosbook as cb

## A one-dimensional flow at full parameter freedom

The chapter's interactive figures step through precomputed parameter values.
Here every slider move recomputes the graph of $f(x)$, its fixed points
(`cb.fixed_points`) and their stability (`cb.stability`): filled dots are
stable, open dots unstable. The flow is the ferromagnet
$\dot x = -x + \tanh((x+m)/T)$; replace `flow` by your own right-hand side.

In [ ]:
def flow(x, T=0.5, m=0.0):
    return -x + np.tanh((x + m) / T)

def explorer(T=0.5, m=0.0):
    x = np.linspace(-1.5, 1.5, 600)
    xs = cb.fixed_points(flow, -1.5, 1.5, T=T, m=m)
    sig = cb.stability(flow, xs, T=T, m=m)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(x, flow(x, T=T, m=m), "k")
    ax.axhline(0, color="gray", linewidth=0.7)
    for xp, s in zip(xs, sig):
        ax.plot(xp, 0, "ko", markerfacecolor="k" if s < 0 else "w", markersize=9)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$f(x)$")
    ax.set_ylim(-1, 1)
    plt.show()
    for xp, s, c in zip(xs, sig, cb.classify(sig)):
        print(f"x* = {xp:+.4f}   sigma = {s:+.4f}   {c}")

interact(explorer,
         T=FloatSlider(0.5, min=0.1, max=2.0, step=0.01),
         m=FloatSlider(0.0, min=-0.5, max=0.5, step=0.01));

## Exercise: hysteresis

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/cont1d.html#exercises). The system is
$\dot x = r x + x^3 - x^5$ with a subcritical bifurcation.

In [ ]:
def f(x, r):
    return r * x + x**3 - x**5

# a)-c): fixed points and their stability for a given r - then do it
# analytically (sympy can solve f = 0 for you) and plot the branches
r = 0.1
xs = cb.fixed_points(f, -1.5, 1.5, r=r)
print(xs, cb.classify(cb.stability(f, xs, r=r)))

# b)/c) as a plot: branches coloured by stability
for rr, xx, stable in cb.fixed_point_branches(f, "r", -0.5, 0.5, -1.5, 1.5):
    plt.plot(rr, xx, "k" if stable else "0.7", linewidth=2)
plt.xlabel("$r$")
plt.ylabel("$x_*$")
plt.show()

In [ ]:
# d) numerical solution for r = 0.3 and x(0) = 0.1; repeat for r = -0.1
sol = solve_ivp(lambda t, x: f(x, 0.3), [0, 20], [0.1], dense_output=True)
t = np.linspace(0, 20, 400)
plt.plot(t, sol.sol(t)[0], "k")
plt.xlabel("$t$")
plt.ylabel("$x(t)$")
plt.show()

In [ ]:
# e)-g): slow sweeps of r. cb.continuation_sweep integrates to a long time
# at each r, starting from the previous end state.
rs = np.linspace(-0.5, 0.5, 201)
up = cb.continuation_sweep(f, "r", rs, 0.1, tinf=100)
# f) Your turn: sweep r downwards from 0.5 to -0.5, starting from up[-1]
down = None
plt.plot(rs, up, "k")
plt.xlabel("$r$")
plt.ylabel("$x_*$")
plt.show()

## Exercise: stability and bifurcations

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/cont1d.html#exercises).

In [ ]:
# a)-c): fixed points of ((nu + 1/2) x + (nu - 1/2) x^3) / (1 + x^2).
# Solve f = 0 with sympy, then classify the stability in terms of nu.
import sympy as sp
x, nu = sp.symbols("x nu")
fs = ((nu + sp.Rational(1, 2)) * x + (nu - sp.Rational(1, 2)) * x**3) / (1 + x**2)
print(sp.solve(sp.Eq(fs, 0), x))
print(sp.simplify(sp.diff(fs, x).subs(x, 0)))

In [ ]:
# d) nu = 1/4: numerical solution from x(0) = 1e-4 and x(0) = 3
def g(x, nu=0.25):
    return ((nu + 0.5) * x + (nu - 0.5) * x**3) / (1 + x**2)

t = np.linspace(0, 40, 800)
for x0 in (1e-4, 3.0):
    sol = solve_ivp(lambda t, x: g(x), [0, 40], [x0], t_eval=t, rtol=1e-8)
    plt.plot(t, sol.y[0], label=f"$x(0)={x0:g}$")
plt.xlabel("$t$")
plt.ylabel("$x(t)$")
plt.legend()
plt.show()

## Exercise: stability and bifurcations (2)

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/cont1d.html#exercises). The system is
$\dot x = -2 - x + r\exp(-x^2)$.

In [ ]:
def f2(x, r):
    return -2 - x + r * np.exp(-x**2)

# a), b): all fixed points for r = 3 and their stability
xs = cb.fixed_points(f2, -5, 5, r=3)
print(xs, cb.stability(f2, xs, r=3), cb.classify(cb.stability(f2, xs, r=3)))

# d): graphs of f for several r reveal the bifurcations in [-10, 10]
x = np.linspace(-4, 3, 400)
for r in (-10, -5, 0, 3, 6, 10):
    plt.plot(x, f2(x, r), label=f"$r={r}$")
plt.axhline(0, color="k", linewidth=0.7)
plt.ylim(-6, 6)
plt.xlabel("$x$")
plt.ylabel("$f_r(x)$")
plt.legend(ncol=2)
plt.show()

# branches x*(r) coloured by stability (c, e, f)
for rr, xx, stable in cb.fixed_point_branches(f2, "r", -10, 10, -4, 3):
    plt.plot(rr, xx, "k" if stable else "0.7", linewidth=2)
plt.xlabel("$r$")
plt.ylabel("$x_*$")
plt.show()

## Exercise: Earth's heat balance

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/cont1d.html#exercises). The temperature obeys
$\dot T = -\epsilon (T+273)^4 + w(T)\cos\theta$ with
$w(T) = a + b\,\mathrm{erf}(cT)$.

In [ ]:
from scipy.special import erf

a, b, c, eps = 4.4e9, 1.3e9, 0.2, 0.5

def w(T):
    return a + b * erf(c * T)

def heat(T, theta):
    return -eps * (T + 273.0)**4 + w(T) * np.cos(theta)

# a): fixed points and stability for theta = 0, 45 and 60 degrees
for deg in (0, 45, 60):
    th = np.radians(deg)
    Ts = cb.fixed_points(heat, -60, 60, n=4001, theta=th)
    sig = cb.stability(heat, Ts, theta=th)
    print(deg, np.round(Ts, 2), cb.classify(sig))

# graph of f for the three latitudes
T = np.linspace(-60, 60, 500)
for deg in (0, 45, 60):
    plt.plot(T, heat(T, np.radians(deg)), label=rf"$\theta={deg}^\circ$")
plt.axhline(0, color="k", linewidth=0.7)
plt.xlabel("$T$")
plt.ylabel("$dT/dt$")
plt.legend()
plt.show()

In [ ]:
# c) potential V(T) = -integral of f, integrated numerically from T = 0
from scipy.integrate import quad

th = np.radians(45)
Tgrid = np.linspace(-60, 60, 121)
V = np.array([-quad(lambda s: heat(s, th), 0, T0)[0] for T0 in Tgrid])
plt.plot(Tgrid, V)
plt.xlabel("$T$")
plt.ylabel("$V(T)$")
plt.show()      # minima of V are the stable fixed points

# d) the time evolution from -100 C and +100 C up to t = 1e-6
for T0 in (-100.0, 100.0):
    sol = solve_ivp(lambda t, T: heat(T, th), [0, 1e-6], [T0], method="LSODA",
                    rtol=1e-8, atol=1e-8, dense_output=True)
    t = np.linspace(0, 1e-6, 400)
    plt.plot(t, sol.sol(t)[0], label=f"$T(0)={T0:g}$")
plt.xlabel("$t$")
plt.ylabel("$T(t)$")
plt.legend()
plt.show()

# e) criterion: f = 0 and f' = 0 at the same time - solve for (T, theta in degrees)
# with fsolve, starting from a guess read off the graphs; try other guesses
from scipy.optimize import fsolve

def fold(v):
    T0, deg = v
    thv = np.radians(deg)
    return [heat(T0, thv), cb.stability(heat, T0, theta=thv)]

print(fsolve(fold, [-5, 38]))
